In [7]:
import sys
import warnings 
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))

import jax
import optax
from flax import nnx
jax.config.update("jax_enable_x64", False)

import pandas as pd
import seaborn as sns
import sklearn.gaussian_process.kernels as gp_kern
from sklearn.gaussian_process import GaussianProcessRegressor

from source.features import PPFeature
from source.models.LaplaceCPR import LaplaceCPR
from source.data_functions import generate_x3_data
from source.visuals import plot_x3_results_multiple
from source.models.MeanFieldBTN import MeanFieldBTN
from source.models.StructPostBTN import StructPostBTN
from source.evaluation import rmse, nll, ecp, wcpi, rce
from source.models.LaplaceBNN import LaplaceBNN, SimpleNN
from source.general_functions import create_dir_if_not_exists

sns.set_theme()
warnings.filterwarnings("ignore")

def train_and_evaluate_models(models: dict[str, object], data):
    x_train, x_test, y_train, _ = data
    results = {}
    for model_name, model in models.items():
        model.fit(x_train, y_train)
        pred_mean, pred_std = model.predict(x_test, return_std=True)
        results[model_name] = {"pred_mean": pred_mean, "pred_std": pred_std}
    return results

### Generate Data:

In [8]:
n_samples, n_samples_test, d_dim, std_err, seed = 20, 100, 1, 3, 13
x_train, y_train, _ = generate_x3_data(n_samples, d_dim, std_err=std_err, seed=seed)
x_test, y_test, y_test_true = generate_x3_data(
    n_samples_test, d_dim, min_v=-5, max_v=5, std_err=std_err, seed=seed+1
)

### Generate Results:

In [9]:
# Shared hyperparameters
beta_e, gamma_w = 1/std_err**2, None
rank, fmap, m_order, n_epoch, n_loss_samples = 2, PPFeature(), 4, 100, 10

models = {
    "LA-BNN": LaplaceBNN(
        SimpleNN(
            in_channels=1, hidden_channels=32, out_channels=1, rngs=nnx.Rngs(0),
        ),
        optax.adam(1e-3),
        batch_size=10,
        n_epoch=3000,
        beta_e=beta_e,
        pred_seed=42,
        curv='full',
        linearized=True,
        verbose=False,
    ),
    "MF-BTN": MeanFieldBTN(
        rank, fmap, m_order, n_epoch, beta_e, gamma_w, seed=seed, 
        opt_params={'train_mode': 'GD', 'lr': 1e-6}, n_loss_samples=n_loss_samples,
    ),
    "SP-BTN": StructPostBTN(
        rank, fmap, m_order, n_epoch, beta_e, gamma_w, seed=seed, 
        opt_params={'train_mode': 'GD', 'lr': 1e-6}, n_loss_samples=n_loss_samples, m_rank=2,
    ),
    "GP": GaussianProcessRegressor(
        kernel=gp_kern.DotProduct()**3 + gp_kern.WhiteKernel(1), 
        random_state=0
    ),
    "LA-TNKM": LaplaceCPR(
        rank, fmap, m_order, n_epoch, beta_e, gamma_w, seed=seed,
        pd_mode='lla', hess_type='last', pd_sample_seed=seed,
    ),
}

data = (x_train, x_test, y_train, y_test)
model2res = train_and_evaluate_models(models, data)

#### Figure with predictive distributions:

In [ ]:
SAVE_DIR = Path(f'./artifacts')
create_dir_if_not_exists(SAVE_DIR)

plot_x3_results_multiple(
    results={model: model2res[model] for model in ["LA-BNN", "SP-BTN", "GP", "LA-TNKM"]}, 
    data=(*data, y_test_true), 
    figsize=(16, 4), 
    y_limits=(-100, 100),
    save_path=SAVE_DIR / 'noisy_cubic_1d.png',
    dpi=1000,
)

#### Table with metrics:

In [11]:
def compute_metrics(
    model2res,
    y_test,
    n_samples_metric=1000,
    alpha=0.9,
    key=jax.random.PRNGKey(0),
) -> pd.DataFrame:
    perc = int(alpha * 100)
    metrics_dict = dict()
    for model_name, res in model2res.items():
        key, subkey = jax.random.split(key)
        p_mean, p_std = res['pred_mean'][:, None], res['pred_std'][:, None]
        # Draw predictive samples: (N, S)
        eps = jax.random.normal(subkey, shape=(len(y_test), n_samples_metric))
        samples = p_mean + eps * p_std
        metrics_dict[model_name] = [
            rmse(y_test, p_mean).item(),
            nll(p_mean, p_std**2, y_test),
            ecp(y_test, samples, alpha=alpha).item(),
            wcpi(samples, alpha=alpha).item(),
            rce(y_test, samples).item(),
        ]
    return pd.DataFrame(
        metrics_dict, 
        index=['RMSE', 'NLL', f'ECP-{perc}', f'WCPI-{perc}', 'RCE']
    ).T.reset_index(names='MODEL')

alpha = 0.95
n_samples_metric = 5000
key = jax.random.PRNGKey(0)
metrics_df = compute_metrics(model2res, y_test, n_samples_metric, alpha, key)

In [ ]:
caption = (
    r"Comparison of predictive performance and uncertainty calibration "
    + r"on the cubic regression task ($y = x^3 + \epsilon $, where $\epsilon \sim \mathcal{N}(0, 3^2)$)"
    + r" for LA-BNN, MF-BTN, SP-BTN, GP, and the proposed LA-TNKM. " 
    + r"Models are trained on 20 data points and evaluated on 100 test points. " 
    + r"Metrics reported include root mean squared error (RMSE), negative log-likelihood (NLL), "
    + r"empirical coverage at 95\% (ECP-95), width of the 95\% prediction interval (WCPI-95), "
    + r"and regression calibration error (RCE). Among the evaluated models, LA-TNKM exhibits "
    + r"predictive behavior more consistent with Gaussian process regression, "
    + r"whereas the other baselines show signs of miscalibration or misspecification."
)
print(
    (
        metrics_df
        .to_latex(
            float_format="%.3f",
            column_format='||l||c|c|c|c|c||',
            caption=caption,
            label="table:x_cube_comparison",
            multirow=False,
            index=False,
        ).replace('_', '-')
    )
);